---
jupyter: python3
format:
  html:
    code-fold: true
    code-tools: true
bibliography: ../../references.bib
author:
  - name: XXXX
    id: dw
    orcid: XXXX
    email: XXXX
    corresponding: true
    degrees: PhD
    affiliation: 
      - name: The Alan Turing Institute
        city: London
        country: United Kingdom
        url: www.turing.ac.uk

---

# Combine census, railspace, and StopsGB

This notebook combines StreetsGB (census), StopsGB (stations), and MapReader (railspace) outputs.


In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
from sklearn.neighbors import BallTree
import pathlib
from utils import country_year_checker

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


### Set Census Year and Country

In [2]:
census_country = "scot" # set to either "EW", "SCOT", or "GB" N.B. Setting to "SCOT" or "GB" means the BBCE Urban Class dataset cannot be used as it's "EW" only.
census_year = 1901

country_year_checker(census_country, census_year, )

In [3]:
def read_streetsgb(streetsgb_path, file_extension, census_country, census_year, geom, ):
    if census_country == "GB" and census_year in [1851, 1861, 1881, 1891, 1901]:
        gdf_list = []
        for country in ["EW", "scot"]:
            gdf = gdf_reader(streetsgb_path,file_extension, geom, country, census_year, )
            gdf["Country"] = country
            # gdf = gdf.rename(columns = {f"{geom}_{country}_{census_year}" : f"{geom}_{census_country}_{census_year}"})
            gdf_list.append(gdf)

        streetsgb_gdf = pd.concat(gdf_list)
        
    elif census_country == "EW" or "scot":
        streetsgb_gdf = gdf_reader(streetsgb_path, file_extension, geom, census_country, census_year, )

    else:
        raise ValueError()
    
    
    streetsgb_gdf = streetsgb_gdf[streetsgb_gdf["geometry"].is_empty == False].reset_index(drop=True).copy()
    return streetsgb_gdf

def gdf_reader(streetsgb_path, file_extension, geom, country, census_year,):
     streetsgb_file = pathlib.Path(streetsgb_path).joinpath(f"{country}_{census_year}_{geom}{file_extension}")
     if streetsgb_file.exists() == False:
         raise ValueError(streetsgb_file)
     else:
         
        # gdf = gpd.read_file(streetsgb_file)
        df = pd.read_csv(streetsgb_file, sep = "\t", usecols=["street_uid", "geometry"], )
        # print(df)
        gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df['geometry']), crs="EPSG:27700")
        # gdf = gdf[[f"{geom}_{country}_{census_year}", "geometry"]].dropna(subset = ["geometry"]).copy() decide if need
     return gdf

## Read and process StopsGB (station data)

Read StopsGB, filter by confidence, cross_ref, ghost_entry, opening and closing year.

In [4]:
stopsgb = pd.read_csv('../data/stopsgb/StopsGB.tsv',sep='\t',usecols=['StationId','Station','Opening','Closing','cross_ref','ghost_entry','conf_station','selected_entity_longitude','selected_entity_latitude'])
stopsgb['Opening_year'] = np.where(stopsgb['Opening'].str[0:4] == 'unkn',np.nan,stopsgb['Opening'].str[0:4])

# Create closing year field from Closing field

stopsgb['Closing_year'] = np.where(stopsgb['Closing'].str[0:4] == 'unkn',np.nan,stopsgb['Closing'].str[0:4])

stopsgb['Opening_year'] = pd.to_numeric(stopsgb['Opening_year'], errors='coerce')
stopsgb['Closing_year'] = pd.to_numeric(stopsgb['Closing_year'], errors='coerce')

# Filter out stations for 1901, filter out low confidence stations, as well as cross references and ghost entries.
stopsgb_filtered = stopsgb[(stopsgb['conf_station'] >= 0.5) 
							& (stopsgb['Opening_year'] <= census_year)
							& ((stopsgb['Closing_year'] > census_year) | (stopsgb['Closing'] == 'still open'))
							& (stopsgb['cross_ref'] == False)
							& (stopsgb['ghost_entry'] == False)]
# print(stopsgb_filtered.shape,stopsgb.shape)


stopsgb_gdf = gpd.GeoDataFrame(stopsgb_filtered,geometry=gpd.points_from_xy(stopsgb_filtered['selected_entity_longitude'], stopsgb_filtered['selected_entity_latitude']),crs='EPSG:4326')

# Reproject for compatibility with RSD boundary dataset
stopsgb_gdf = stopsgb_gdf.to_crs('EPSG:27700')

stopsgb_gdf = stopsgb_gdf[['StationId','Station','geometry']]

print(stopsgb_gdf.shape)

(5374, 3)


## Read and process MapReader output (railway track + building data)

In [5]:
# Read rail patches
mapreader_rail = pd.read_csv("../data/mapreader/MapReader_Data_SIGSPATIAL_2022/outputs/label_01_03/pred_01_03_keep_01_0250.csv",sep=',',usecols=['center_lon','center_lat','pred','conf'])
mapreader_rail_gdf = gpd.GeoDataFrame(mapreader_rail,geometry=gpd.points_from_xy(mapreader_rail['center_lon'], mapreader_rail['center_lat']),crs='EPSG:4326')
mapreader_rail_gdf = mapreader_rail_gdf.to_crs('EPSG:27700')
mapreader_rail_gdf['coords1'] = mapreader_rail_gdf['geometry'].x
mapreader_rail_gdf['coords2'] = mapreader_rail_gdf['geometry'].y

# Read all patches
mapreader_all = pd.read_csv("../data/mapreader/MapReader_Data_SIGSPATIAL_2022/outputs/patches_all.csv",sep=',',usecols=['center_lon','center_lat'])
mapreader_all_gdf = gpd.GeoDataFrame(mapreader_all,geometry=gpd.points_from_xy(mapreader_all['center_lon'], mapreader_all['center_lat']),crs='EPSG:4326')
mapreader_all_gdf = mapreader_all_gdf.to_crs('EPSG:27700')
mapreader_all_gdf['coords1'] = mapreader_all_gdf['geometry'].x
mapreader_all_gdf['coords2'] = mapreader_all_gdf['geometry'].y



KeyboardInterrupt: 

In [ ]:
X_railspace = np.array(list(zip(mapreader_rail_gdf['coords1'],mapreader_rail_gdf['coords2'])))
print(X_railspace.shape)
X_all = np.array(list(zip(mapreader_all_gdf['coords1'],mapreader_all_gdf['coords2'])))
print(X_all.shape)

(483278, 2)
(30490411, 2)


In [7]:
leafsize = 40
tree_railspace = BallTree(X_railspace, leaf_size=leafsize,metric='euclidean')
tree_all = BallTree(X_all, leaf_size=leafsize,metric='euclidean')

In [8]:
#this needs fixing because the street uids are not unique for GB.

if census_country == "GB":
    blockcols = ["street_uid", "Country"]
else:
    blockcols = ["street_uid"]
def get_nearest_station(stopsgb, streetsgb, distance_col, geom, census_country, census_year, ):

    dist2stopsgb_tmp = gpd.sjoin_nearest(stopsgb,streetsgb,how='right',distance_col='distance').drop(columns = ["index_left"]) # calcs nearest StopsGB station to each street
    dist2stopsgb_multi_only = dist2stopsgb_tmp[dist2stopsgb_tmp.duplicated(subset=blockcols, keep = False)]
    dist2stopsgb_multi_only = dist2stopsgb_multi_only.groupby(blockcols)["StationId"].apply(lambda x: x.to_list()).reset_index(name = "equidistant_stationids") # keep record of the stations that are equidistant to a street
    dist2stopsgb = pd.merge(left = dist2stopsgb_tmp, right = dist2stopsgb_multi_only, on = blockcols, how="left")

    dist2stopsgb = dist2stopsgb.drop_duplicates(subset= blockcols, keep = "first").copy() # BE AWARE THAT IF NOT GB THEN COUNTRY NEEDS TO BE REMOVED FROM THIS.

    return dist2stopsgb.reset_index(drop=True)

def create_coords_field(df, geometry_field, new_coords_field, ):

    parse = lambda x: re.compile(r'\({1,2}(.*?)\)').findall(str(x)) # modified from Kaspar's by adding str() to x
    to_tuple = lambda x: tuple(map(float,x.split(' ')))
    to_coords = lambda r: [to_tuple(x) for i in r for x in i.split(', ')]

    df[new_coords_field] = df[geometry_field].apply(parse)
    df[new_coords_field] = df[new_coords_field].apply(to_coords)
    df[new_coords_field] = df[new_coords_field].apply(lambda r: [[x[0], x[1] ] for x in r])

    return df

In [9]:
def rail_density(x,tree_rail,tree_all,target_radius):
    #could add nan capture here rather than removing from whole dataframe
    # print(x)
    avg_rail =  tree_rail.query_radius(x, r=target_radius,count_only=True)
    avg_all = tree_all.query_radius(x, r=target_radius,count_only=True)

    # may need to sort out nans


    
    mean_val = np.mean(avg_rail / avg_all)
    stddev_val = np.std(avg_rail / avg_all)

    if mean_val == np.nan:
        print("meanval", x)
    elif stddev_val == np.nan:
        print("stdval", x)
    # return np.mean(avg_rail / avg_all),np.std(avg_rail / avg_all)
    return mean_val, stddev_val


def get_railspace_scores(streets_gdf, radius, railspace_patch_tree, all_patch_tree, ):
    streets_gdf[f'rail_density_{radius}m'], streets_gdf[f'rail_density_{radius}m_std'] = zip(*streets_gdf['coords'].apply(rail_density, 
                                                        tree_rail = railspace_patch_tree, 
                                                        tree_all = all_patch_tree,
                                                        target_radius = radius))
    return streets_gdf

## Read and process StreetsGB (census data) and calculate nearest station and railspace density

In [10]:

public_release = True

target_geometries = ["gb1900", "osopenroads"] #calculate distances and densities for the 2 different geometries used in StreetsGB

density_distances = [100, ] # set distances from street to calculate railspace densities (in meters)

for geom in target_geometries:
    streetsgb_gdf = read_streetsgb(f"../data/streetsgb/", ".tsv", census_country, census_year, geom, )
    # print(streetsgb_gdf)
    print(geom)

    dist2stopsgb = get_nearest_station(stopsgb_gdf, streetsgb_gdf, "distance", geom, census_country, census_year, )
    dist2stopsgb = create_coords_field(dist2stopsgb, "geometry", "coords", )

    # calc railspace densities at set distances
    for rad in density_distances:
        print(rad)
        dist2stopsgb = get_railspace_scores(dist2stopsgb, rad, tree_railspace, tree_all, )

    dist2stopsgb = dist2stopsgb.drop(columns = ["coords"])

    if public_release == True:
        dist2stopsgb = dist2stopsgb.drop(columns = ["geometry"])


    if census_country == "GB":
        col_order = ["street_uid", "Country", "StationId", "equidistant_stationids", "distance", "rail_density_100m", "rail_density_100m_std"]
    else:
        col_order = ["street_uid", "StationId", "equidistant_stationids", "distance", "rail_density_100m", "rail_density_100m_std"]

    dist2stopsgb[col_order].to_csv(f'../data/outputs/streetsgb_enhanced/{geom}_{census_country}_{census_year}_stations_and_raildensity.tsv',sep='\t',index=False)

gb1900
100
osopenroads
100


In [ ]:
output_file[icols] = output_file[icols].apply(pd.to_numeric, downcast = "integer" )
output_file[fcols] = output_file[fcols].apply(pd.to_numeric, downcast = "float" )